In [25]:
from pathlib import Path

import pandas as pd
import numpy as np
import zipfile
import re
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 300)

PROJECT_DIR = Path("..").resolve()

PUBLIC_RAW = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "raw"
)

INTERIM_DIR = (
    PROJECT_DIR
    / "data"
    / "public_v2"
    / "interim"
)

REPORTS_DIR = (
    PROJECT_DIR
    / "reports"
    / "tables"
)

DASH_DIR = (
    PUBLIC_RAW
    / "08_dashboards"
)

INTERIM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [26]:
dash_main = pd.read_csv(
    DASH_DIR
    / "dependent and independent variables.csv"
)

print(
    "Shape:",
    dash_main.shape
)

print(
    "Columnas:"
)

print(
    dash_main.columns.tolist()
)

display(
    dash_main.head(20)
)

Shape: (1008, 16)
Columnas:
['P', 'DB', 'Issue', 'Type', 'Q', 'Answer', 'Time', 'SGLT', 'mental', 'physical', 'temporal', 'performance', 'effort', 'frustration', 'avg_others', 'avg_all']


,P,DB,Issue,Type,Q,Answer,Time,SGLT,mental,physical,temporal,performance,effort,frustration,avg_others,avg_all
0,f118ebdd7e9350d704d9f91ee7de2702,3,Order&Grouping,Baseline,1,0,5.33,0,14.0,10.0,6,14,10,14.0,10.8,11.33
1,f118ebdd7e9350d704d9f91ee7de2702,2,InforOverload,Intervention,1,1,8.53,0,6.0,2.0,4,20,4,10.0,5.2,7.67
2,f118ebdd7e9350d704d9f91ee7de2702,1,InforOverload,Baseline,1,0,2.25,0,1.0,1.0,1,20,0,0.0,0.6,3.83
3,f118ebdd7e9350d704d9f91ee7de2702,3,Order&Grouping,Baseline,2,0,7.85,0,10.0,10.0,6,20,10,18.0,10.8,12.33
4,f118ebdd7e9350d704d9f91ee7de2702,7,DataUnderstanding,Baseline,2,1,8.82,0,4.0,4.0,2,8,5,14.0,5.8,6.17
5,f118ebdd7e9350d704d9f91ee7de2702,4,Order&Grouping,Intervention,2,1,8.67,0,6.0,10.0,2,20,4,3.0,5.0,7.50
6,f118ebdd7e9350d704d9f91ee7de2702,2,InforOverload,Intervention,2,1,5.80,0,16.0,13.0,6,20,6,10.0,10.2,11.83
7,f118ebdd7e9350d704d9f91ee7de2702,5,DataPresentation,Baseline,1,1,4.15,0,8.0,8.0,6,12,7,11.0,8.0,8.67
8,f118ebdd7e9350d704d9f91ee7de2702,6,DataPresentation,Intervention,1,1,2.10,0,2.0,2.0,2,20,2,0.0,1.6,4.67
9,f118ebdd7e9350d704d9f91ee7de2702,7,DataUnderstanding,Baseline,1,0,4.43,0,4.0,4.0,3,10,3,10.0,4.8,5.67


In [27]:
print(
    "Participantes:",
    dash_main["P"].nunique()
)

print(
    "Dashboards:",
    sorted(
        dash_main["DB"]
        .dropna()
        .unique()
    )
)

print(
    "Preguntas:",
    sorted(
        dash_main["Q"]
        .dropna()
        .unique()
    )
)

print(
    "\nIssue:"
)

print(
    dash_main[
        "Issue"
    ].value_counts(
        dropna=False
    )
)

print(
    "\nType:"
)

print(
    dash_main[
        "Type"
    ].value_counts(
        dropna=False
    )
)

Participantes: 63
Dashboards: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Preguntas: [np.int64(1), np.int64(2)]

Issue:
Issue
Order&Grouping       252
InforOverload        252
DataUnderstanding    252
DataPresentation     252
Name: count, dtype: int64

Type:
Type
Baseline        504
Intervention    504
Name: count, dtype: int64


In [28]:
candidate_key = [
    "P",
    "DB",
    "Q"
]

duplicate_count = (
    dash_main
    .duplicated(
        subset=candidate_key
    )
    .sum()
)

print(
    "Duplicados P+DB+Q:",
    duplicate_count
)

Duplicados P+DB+Q: 0


In [29]:
key_counts = (
    dash_main
    .groupby(
        candidate_key
    )
    .size()
)

print(
    key_counts.value_counts()
)

1    1008
Name: count, dtype: int64


In [30]:
workbook_path = (
    DASH_DIR
    / "Interaction Patterns Dataset.xlsx"
)

xls = pd.ExcelFile(
    workbook_path
)

print(
    xls.sheet_names
)

['BENCHMARK', 'D1-Q1', 'D2-Q1', 'D3-Q2', 'D4-Q2', 'D5-Q1', 'D6-Q1', 'D7-Q2', 'D8-Q2', 'Event-element further explanati']


In [31]:
for sheet in xls.sheet_names:

    print(
        "\n",
        "=" * 100
    )

    print(
        "SHEET:",
        sheet
    )

    raw_sheet = pd.read_excel(
        workbook_path,
        sheet_name=sheet,
        header=None,
        dtype=str,
        nrows=15
    )

    print(
        "Shape preview:",
        raw_sheet.shape
    )

    display(
        raw_sheet
    )


SHEET: BENCHMARK
Shape preview: (15, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,Minsup,Max gap,Max pattern length,Avg length,# of patterns,Max Minsup,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D1-Q1,NaN,NaN,NaN,NaN,NaN,NaN,All Dashboards,NaN,NaN,NaN,NaN,NaN,NaN,Key,NaN
2,0.5,4,5,2.71022727272727,176,39,NaN,Minsup,Max gap,Avg Max pattern length,Avg length,Avg # of patterns,Avg Max minsup,NaN,Minsup,"The minimum support value (minimum threshold of occurrence frequency of patterns in the dataset). For example, 0.5 means the algorithm will return patterns that are exhibited by at least 50% of the participants."
3,0.55,4,5,2.53968253968254,126,39,NaN,0.5,4,7.25,3.462989317,168.125,36.5,NaN,Max gap,"The maximum allowed gap between consecutive events in a pattern. For example, a max gap of 4 means that at most 3 events are allowed to exist between two conecutive events in a found pattern (gap of N means at most N-1 items are allowed)."
4,0.6,4,5,2.43,91,42,NaN,0.55,4,6.625,3.245033202,120.625,39.75,NaN,Max pattern length,"The maximum pattern length in the found patterns. We do not impose a maximum length; here we just report what we found to help us decide which minimum support value to go with. For example, if the algorithm found 5 patterns with lengths: 6, 4, 5, 7, 4 then the max pattern length = 7."
5,0.7,4,5,2.21,52,48,NaN,0.6,4,6.375,3.110881988,96.125,41.5,NaN,Avg length,"The average length of the found patterns.Here too, we just report this value so we can make an informed decision. For example, if the algorithm found 5 patterns with lengths: 6, 4, 5, 7, 4 then the avg length = 5.2."
6,0.8,4,3,2,37,48,NaN,0.7,4,5.75,2.761550805,57.875,44.625,NaN,# of patterns,"The number of patterns returned by the algorithm when setting minsup and max gap at their declared values. For example, if the algorithm found 5 patterns, then # of patterns would simply be 5."
7,D2-Q1,NaN,NaN,NaN,NaN,NaN,NaN,0.8,4,4.25,2.460811688,34.25,48.375,NaN,Max minsup,"When running the algorithm with a set minsup and max gap values, it will return several patterns where each pattern has a minsup value (meaning this pattern was exhibited by X number of participants). Max minsup means the maximum minsup value for all the returned patterns. For example, if the al..."
8,0.5,4,6,2.68141592920354,113,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"These statistics are reported for all of the dashboards collectively in cells H2:M8. Eventually, we used minsup=0.55 and max gap=4. All results here are yielded with these parameters.",NaN
9,0.55,4,5,2.42105263157894,76,44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



SHEET: D1-Q1
Shape preview: (15, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,"Pattern (0.55, gap=4)",Support,Split,NaN,NaN,NaN,NaN,Abstracted,NaN,NaN,NaN,NaN,Length,Key,NaN,NaN
1,"28, 109, 26, 28",38,28,109,26,28,NaN,expl country filter,use country filter,use country filter,expl country filter,NaN,4,Code,Event-element,Meaning
2,"12, 30, 18, 19, 20",35,12,30,18,19,20,expl view filter,filter globally,expl view filter,use view filter,expl view filter,5,28,mhover-bs-select,explore country filter
3,"12, 30, 19, 20",32,12,30,19,20,NaN,expl view filter,filter globally,use view filter,expl view filter,NaN,4,12,mhover-div_test,explore view filter
4,"6, 6, 6, 6",31,6,6,6,6,NaN,expl page,expl page,expl page,expl page,NaN,4,6,hover-DashboardPage,explore page
5,"28, 28, 109, 26",28,28,28,109,26,NaN,expl country filter,expl country filter,use country filter,use country filter,NaN,4,4,mhover-DashboardPage,explore page
6,"4, 16, 115, 29",28,4,16,115,29,NaN,expl page,goto Q,read db title,read aTitle,NaN,4,9,hover-_DIV1_DIV1_DIV1_SECTION1_DIV3_DIV1_DIV1_DIV2,goto Q
7,"9, 4, 16, 115",28,9,4,16,115,NaN,goto Q,expl page,goto Q,read db title,NaN,4,42,hover-_DIV1_DIV1_DIV1_SECTION1_DIV3_DIV3_DIV1_DIV1_BUTTON1,explore country filter
8,"42, 23, 24, 23",28,42,23,24,23,NaN,expl country filter,expl country filter,use country filter,expl country filter,NaN,4,109,click-bs-select,use country filter
9,"28, 28, 26, 28",28,28,28,26,28,NaN,expl country filter,expl country filter,use country filter,expl country filter,NaN,4,30,click-div_test,filter globally



SHEET: D2-Q1
Shape preview: (15, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,"Pattern (0.55, gap=4)",Support,Split,NaN,NaN,NaN,NaN,Abstracted,NaN,NaN,NaN,NaN,Length,Key,NaN,NaN
1,"31, 48, 49, 50, 51",37,31,48,49,50,51,expl view filter,filter globally,expl view filter,use view filter,expl view filter,5,Code,Event-element,Meaning
2,"12, 39, 87, 88",33,12,39,87,88,NaN,expl page,goto Q,read db title,read aTitle,NaN,4,31,mhover-div_test,explore view filter
3,"14, 12, 39, 87",30,14,12,39,87,NaN,read aNumOrPercent,expl page,goto Q,read db title,NaN,4,12,mhover-DashboardPage,explore page
4,"3, 14, 12, 39",29,3,14,12,39,NaN,goto Q,read aNumOrPercent,expl page,goto Q,NaN,4,14,hover-aNumOrPercent,read aNumOrPercent
5,"31, 31, 49, 50, 51",29,31,31,49,50,51,expl view filter,expl view filter,expl view filter,use view filter,expl view filter,5,3,hover-_DIV1_DIV1_DIV1_SECTION1_DIV3_DIV1,goto Q
6,"3, 32, 12, 39",28,3,32,12,39,NaN,goto Q,goto Q,expl page,goto Q,NaN,4,29,hover-_DIV1_DIV1_DIV1_SECTION1_DIV3_DIV3_DIV1_DIV1_BUTTON1,explore country filter
7,"29, 61, 62, 63, 64",28,29,61,62,63,64,expl country filter,expl country filter,use country filter,use age filter,expl country filter,5,48,click-div_test,filter globally
8,"48, 31, 49, 50, 51",28,48,31,49,50,51,filter globally,expl view filter,expl view filter,use view filter,expl view filter,5,39,hover-_DIV1_DIV1_DIV1_SECTION1_DIV2,goto Q
9,"31, 48, 31, 50, 51",28,31,48,31,50,51,expl view filter,filter globally,expl view filter,use view filter,expl view filter,5,32,hover-_DIV1_DIV1_DIV1_SECTION1_DIV3_DIV1_DIV1_DIV2_DIV1_DIV2,goto Q


... (10 outputs left out)


SHEET: D8-Q2
Shape preview: (15, 22)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21
0,"Pattern (0.55, gap=4)",Support,Split,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Length,Abstracted,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Key,NaN,NaN
1,"1, 1, 5, 1",38,1,1,5,1,NaN,NaN,NaN,NaN,4,expl page,expl page,expl continent filter,expl page,NaN,NaN,NaN,NaN,Code,Event-element,Meaning
2,"15, 13, 14, 15",37,15,13,14,15,NaN,NaN,NaN,NaN,4,expl custom selector,use custom selector,use custom selector,expl custom selector,NaN,NaN,NaN,NaN,1,hover-DashboardPage,expl page
3,"30, 4, 1, 1",37,30,4,1,1,NaN,NaN,NaN,NaN,4,read ch1,custom ch1,expl page,expl page,NaN,NaN,NaN,NaN,15,mhover-id1_d8b,expl custom selector
4,"16, 21, 29, 16, 16",36,16,21,29,16,16,NaN,NaN,NaN,5,expl country filter,use country filter,use country filter,expl country filter,expl country filter,NaN,NaN,NaN,30,hover-frame_d8b,read ch1
5,"4, 1, 5, 1, 6",36,4,1,5,1,6,NaN,NaN,NaN,5,custom ch1,expl page,expl continent filter,expl page,read db title,NaN,NaN,NaN,16,mhover-bs-select,expl country filter
6,"1, 1, 1, 6, 1",36,1,1,1,6,1,NaN,NaN,NaN,5,expl page,expl page,expl page,read db title,expl page,NaN,NaN,NaN,4,hover-_DIV1_DIV1_DIV1_SECTION1_DIV4_DIV1_DIV1,customise ch1
7,"4, 5, 1, 6",35,4,5,1,6,NaN,NaN,NaN,NaN,4,custom ch1,expl continent filter,expl page,read db title,NaN,NaN,NaN,NaN,3,mhover-line_d8b,read ch1
8,"1, 1, 6, 19, 1",35,1,1,6,19,1,NaN,NaN,NaN,5,expl page,expl page,read db title,read db title,expl page,NaN,NaN,NaN,5,hover-Continent,expl continent filter
9,"1, 5, 1, 6, 1",35,1,5,1,6,1,NaN,NaN,NaN,5,expl page,expl continent filter,expl page,read db title,expl page,NaN,NaN,NaN,8,mhover-Continent,expl continent filter



SHEET: Event-element further explanati
Shape preview: (15, 2)


,0,1
0,Event-element,Further explanation
1,country filter,Country filter is used to filter the countries that are displayed on the dashboard.
2,view filter,"View filter is used to filter the view that the user is seeing and interacting with on the dashboard. For example, it can switch the view between those who received one dose of vaccination or those who received two doses."
3,explore page,Page is the background page of the dashboard.
4,goto Q,"Since the task question of the experiment was displayed on the same dashboard page, this event represents going to that question."
5,filter globally,NaN
6,read aTitle,"Reading any title or subtitle in the dashboard (e.g., a title of a chart)."
7,read db title,Reading the title of the dashboard.
8,read aNumOrPercent,Reading any specific number of percentage displayed on the dashboard.
9,customise ch1,"Any local customisation event of the top left chart. Customisation events include expanding the chart to zoom on it, viewing data in raw format (tables), switching to an alternative type of chart, hiding a chart, removing its colours, filtering locally, searching within the data, downloading an ..."


In [32]:
participant_ids = set(
    dash_main[
        "P"
    ]
    .dropna()
    .astype(str)
)

participant_matches = []

for sheet in xls.sheet_names:

    temp = pd.read_excel(
        workbook_path,
        sheet_name=sheet,
        header=None,
        dtype=str
    )

    for row_idx in range(
        len(temp)
    ):

        for col_idx in range(
            temp.shape[1]
        ):

            value = temp.iat[
                row_idx,
                col_idx
            ]

            if (
                pd.notna(value)
                and
                str(value).strip()
                in participant_ids
            ):

                participant_matches.append({
                    "sheet":
                        sheet,

                    "row":
                        row_idx,

                    "column":
                        col_idx,

                    "participant":
                        str(value).strip()
                })

In [33]:
participant_match_df = pd.DataFrame(
    participant_matches
)

print(
    "Coincidencias de participantes:",
    len(
        participant_match_df
    )
)

if not participant_match_df.empty:

    print(
        "Participantes distintos encontrados:",
        participant_match_df[
            "participant"
        ].nunique()
    )

    display(
        participant_match_df.head(100)
    )

Coincidencias de participantes: 0


In [34]:
if not participant_match_df.empty:

    print(
        participant_match_df[
            "sheet"
        ].value_counts()
    )

In [35]:
dash_zip = (
    DASH_DIR
    / "Interaction Data.zip"
)

with zipfile.ZipFile(
    dash_zip,
    "r"
) as z:

    real_jsons = [
        name
        for name in z.namelist()
        if (
            name.endswith(".json")
            and
            "__MACOSX"
            not in name
            and
            not Path(name).name.startswith(
                "._"
            )
        )
    ]

print(
    "Logs reales:",
    len(real_jsons)
)

Logs reales: 50


In [36]:
log_registry = pd.DataFrame({
    "source_file":
        [
            Path(name).name
            for name in real_jsons
        ]
})

log_registry[
    "sessionstart_from_filename"
] = pd.to_numeric(
    log_registry[
        "source_file"
    ]
    .str.replace(
        ".json",
        "",
        regex=False
    ),
    errors="coerce"
)

display(
    log_registry.head(20)
)

print(
    log_registry.shape
)

,source_file,sessionstart_from_filename
0,1633946848251.json,1633946848251
1,1634049516613.json,1634049516613
2,1635153366300.json,1635153366300
3,1634637167060.json,1634637167060
4,1633852258325.json,1633852258325
5,1636971248412.json,1636971248412
6,1635877381217.json,1635877381217
7,1635529899496.json,1635529899496
8,1634902121403.json,1634902121403
9,1632586286250.json,1632586286250


(50, 2)


In [37]:
session_ids = set(
    log_registry[
        "sessionstart_from_filename"
    ]
    .dropna()
    .astype("int64")
    .astype(str)
)

session_matches = []

for sheet in xls.sheet_names:

    temp = pd.read_excel(
        workbook_path,
        sheet_name=sheet,
        header=None,
        dtype=str
    )

    for row_idx in range(
        len(temp)
    ):

        for col_idx in range(
            temp.shape[1]
        ):

            value = temp.iat[
                row_idx,
                col_idx
            ]

            if pd.isna(value):
                continue

            clean_value = (
                str(value)
                .strip()
                .replace(
                    ".0",
                    ""
                )
            )

            if clean_value in session_ids:

                session_matches.append({
                    "sheet":
                        sheet,

                    "row":
                        row_idx,

                    "column":
                        col_idx,

                    "sessionstartms":
                        clean_value
                })

In [38]:
session_match_df = pd.DataFrame(
    session_matches
)

print(
    "Session IDs encontrados en workbook:",
    len(
        session_match_df
    )
)

if not session_match_df.empty:

    print(
        "Sesiones distintas:",
        session_match_df[
            "sessionstartms"
        ].nunique()
    )

    display(
        session_match_df.head(100)
    )

Session IDs encontrados en workbook: 0


In [39]:
participant_log_matches = []

with zipfile.ZipFile(
    dash_zip,
    "r"
) as z:

    for name in real_jsons:

        raw = z.read(
            name
        )

        text = raw.decode(
            "utf-8",
            errors="ignore"
        )

        found = [
            participant
            for participant
            in participant_ids
            if participant in text
        ]

        for participant in found:

            participant_log_matches.append({
                "source_file":
                    Path(name).name,

                "participant":
                    participant
            })

In [40]:
participant_log_match_df = pd.DataFrame(
    participant_log_matches
)

print(
    "Matches P dentro de logs:",
    len(
        participant_log_match_df
    )
)

if not participant_log_match_df.empty:

    display(
        participant_log_match_df
    )

Matches P dentro de logs: 0


In [41]:
join_audit = pd.DataFrame([
    {
        "test":
            "P_IN_WORKBOOK",

        "matches":
            len(
                participant_match_df
            ),

        "unique_matches":
            (
                participant_match_df[
                    "participant"
                ].nunique()
                if not participant_match_df.empty
                else 0
            )
    },

    {
        "test":
            "SESSIONSTART_IN_WORKBOOK",

        "matches":
            len(
                session_match_df
            ),

        "unique_matches":
            (
                session_match_df[
                    "sessionstartms"
                ].nunique()
                if not session_match_df.empty
                else 0
            )
    },

    {
        "test":
            "P_IN_RAW_LOGS",

        "matches":
            len(
                participant_log_match_df
            ),

        "unique_matches":
            (
                participant_log_match_df[
                    "participant"
                ].nunique()
                if not participant_log_match_df.empty
                else 0
            )
    }
])

join_audit

,test,matches,unique_matches
0,P_IN_WORKBOOK,0,0
1,SESSIONSTART_IN_WORKBOOK,0,0
2,P_IN_RAW_LOGS,0,0


In [42]:
OBJECT_ID_RE = re.compile(
    r'ObjectId\("([^"]+)"\)'
)

ISO_DATE_RE = re.compile(
    r'ISODate\("([^"]+)"\)'
)

NUMBER_LONG_RE = re.compile(
    r'NumberLong\("?(-?\d+)"?\)'
)


def parse_mongo_line(line):

    line = line.strip()

    if not line:
        return None

    line = OBJECT_ID_RE.sub(
        r'"\1"',
        line
    )

    line = ISO_DATE_RE.sub(
        r'"\1"',
        line
    )

    line = NUMBER_LONG_RE.sub(
        r'\1',
        line
    )

    return json.loads(
        line
    )

In [43]:
event_samples = []

wanted_events = {
    "mouseup",
    "mousedown",
    "keypress",
    "keydown",
    "scroll",
    "change",
    "dblclick"
}

with zipfile.ZipFile(
    dash_zip,
    "r"
) as z:

    for name in real_jsons[:5]:

        with z.open(name) as f:

            for raw_line in f:

                text = raw_line.decode(
                    "utf-8-sig",
                    errors="replace"
                )

                obj = parse_mongo_line(
                    text
                )

                if obj is None:
                    continue

                event = obj.get(
                    "event"
                )

                if event not in wanted_events:
                    continue

                event_samples.append({
                    "source_file":
                        Path(name).name,

                    "event":
                        event,

                    "timestampms":
                        obj.get(
                            "timestampms"
                        ),

                    "episodeCount":
                        obj.get(
                            "episodeCount"
                        ),

                    "urlFull":
                        obj.get(
                            "urlFull"
                        ),

                    "mouse":
                        obj.get(
                            "mouse"
                        ),

                    "key":
                        obj.get(
                            "key"
                        ),

                    "scroll":
                        obj.get(
                            "scroll"
                        ),

                    "node":
                        obj.get(
                            "node"
                        ),

                    "change":
                        obj.get(
                            "change"
                        )
                })

                if len(
                    event_samples
                ) >= 100:
                    break

        if len(
            event_samples
        ) >= 100:
            break

In [44]:
event_sample_df = pd.DataFrame(
    event_samples
)

display(
    event_sample_df.head(100)
)

,source_file,event,timestampms,episodeCount,urlFull,mouse,key,scroll,node,change
0,1633946848251.json,scroll,1633946856055,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,None,NaN,"{'clientHeight': 1361, 'scrollHeight': 1628, 'scrollY': 11, 'scrollYPercentage': 84}",None,None
1,1633946848251.json,scroll,1633946856254,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,None,NaN,"{'clientHeight': 1361, 'scrollHeight': 1628, 'scrollY': 168, 'scrollYPercentage': 94}",None,None
2,1633946848251.json,scroll,1633946856454,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,None,NaN,"{'clientHeight': 1361, 'scrollHeight': 1628, 'scrollY': 267, 'scrollYPercentage': 100}",None,None
3,1633946848251.json,mousedown,1633946858363,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 28, 'coordY': 1211, 'offsetX': 13, 'offsetY': 1211, 'but': 'l'}",NaN,None,"{'dom': 'BODY/DIV[1]/DIV[1]/DIV[1]/DIV[1]', 'class': ['container-fluid'], 'type': 'DIV', 'textContent': ' '}",None
4,1633946848251.json,mouseup,1633946858472,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 28, 'coordY': 1211, 'offsetX': 13, 'offsetY': 1211, 'but': 'l'}",NaN,None,"{'dom': 'BODY/DIV[1]/DIV[1]/DIV[1]/DIV[1]', 'class': ['container-fluid'], 'type': 'DIV', 'textContent': ' '}",None
5,1633946848251.json,mousedown,1633946859015,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 32, 'coordY': 1207, 'offsetX': 2, 'offsetY': 4, 'but': 'l'}",NaN,None,"{'dom': 'id(""consent1"")/DIV[1]/DIV[1]/LABEL[1]/INPUT[1]', 'id': 'consent1', 'inheritedId': True, 'name': 'consent1', 'class': [''], 'type': 'INPUT', 'textContent': 'null', 'textValue': 'c1 done'}",None
6,1633946848251.json,mouseup,1633946859125,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 32, 'coordY': 1207, 'offsetX': 2, 'offsetY': 4, 'but': 'l'}",NaN,None,"{'dom': 'id(""consent1"")/DIV[1]/DIV[1]/LABEL[1]/INPUT[1]', 'id': 'consent1', 'inheritedId': True, 'name': 'consent1', 'class': [''], 'type': 'INPUT', 'textContent': 'null', 'textValue': 'c1 done'}",None
7,1633946848251.json,mousedown,1633946859600,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 36, 'coordY': 1229, 'offsetX': 6, 'offsetY': 6, 'but': 'l'}",NaN,None,"{'dom': 'id(""consent1"")/DIV[1]/DIV[2]/LABEL[1]/INPUT[1]', 'id': 'consent1', 'inheritedId': True, 'name': 'consent1', 'class': [''], 'type': 'INPUT', 'textContent': 'null', 'textValue': 'c2 done'}",None
8,1633946848251.json,mouseup,1633946859754,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 36, 'coordY': 1229, 'offsetX': 6, 'offsetY': 6, 'but': 'l'}",NaN,None,"{'dom': 'id(""consent1"")/DIV[1]/DIV[2]/LABEL[1]/INPUT[1]', 'id': 'consent1', 'inheritedId': True, 'name': 'consent1', 'class': [''], 'type': 'INPUT', 'textContent': 'null', 'textValue': 'c2 done'}",None
9,1633946848251.json,mousedown,1633946860219,1,http://ec2-18-170-45-231.eu-west-2.compute.amazonaws.com/study/shinyapp/,"{'coordX': 40, 'coordY': 1250, 'offsetX': 10, 'offsetY': 7, 'but': 'l'}",NaN,None,"{'dom': 'id(""consent1"")/DIV[1]/DIV[3]/LABEL[1]/INPUT[1]', 'id': 'consent1', 'inheritedId': True, 'name': 'consent1', 'class': [''], 'type': 'INPUT', 'textContent': 'null', 'textValue': 'c3 done'}",None


In [45]:
participant_match_df.to_csv(
    REPORTS_DIR
    / "dashboards_participant_workbook_matches.csv",
    index=False
)

session_match_df.to_csv(
    REPORTS_DIR
    / "dashboards_session_workbook_matches.csv",
    index=False
)

participant_log_match_df.to_csv(
    REPORTS_DIR
    / "dashboards_participant_log_matches.csv",
    index=False
)

join_audit.to_csv(
    REPORTS_DIR
    / "dashboards_join_audit.csv",
    index=False
)

event_sample_df.to_json(
    INTERIM_DIR
    / "dashboards_event_sample.json",
    orient="records",
    indent=2
)

In [46]:
expected = [
    REPORTS_DIR
    / "dashboards_participant_workbook_matches.csv",

    REPORTS_DIR
    / "dashboards_session_workbook_matches.csv",

    REPORTS_DIR
    / "dashboards_participant_log_matches.csv",

    REPORTS_DIR
    / "dashboards_join_audit.csv",

    INTERIM_DIR
    / "dashboards_event_sample.json"
]

for path in expected:

    print(
        path.name,
        "✅" if path.exists()
        else "❌"
    )

dashboards_participant_workbook_matches.csv ✅
dashboards_session_workbook_matches.csv ✅
dashboards_participant_log_matches.csv ✅
dashboards_join_audit.csv ✅
dashboards_event_sample.json ✅


In [47]:
rows_per_participant = (
    dash_main
    .groupby("P")
    .size()
)

print(
    rows_per_participant.value_counts()
)

assert (
    len(dash_main)
    ==
    dash_main["P"].nunique()
    * 8
    * 2
)

print(
    "Diseño experimental:",
    f'{dash_main["P"].nunique()} participantes × 8 DB × 2 Q = {len(dash_main)} filas ✅'
)

16    63
Name: count, dtype: int64
Diseño experimental: 63 participantes × 8 DB × 2 Q = 1008 filas ✅


In [48]:
dashboards_join_decision = pd.DataFrame([
    {
        "dataset": "Dashboards",
        "experimental_rows": len(dash_main),
        "participants": dash_main["P"].nunique(),
        "raw_log_files": len(real_jsons),
        "P_in_workbook": len(participant_match_df),
        "sessionstart_in_workbook": len(session_match_df),
        "P_in_raw_logs": len(participant_log_match_df),
        "join_status": "NO_REPRODUCIBLE_JOIN",
        "supervised_use": False,
        "dataset_role": "X_ONLY_AUXILIARY",
        "reason":
            "No existe una clave observable que permita asociar cada log "
            "con P+DB+Q sin inferir correspondencias."
    }
])

dashboards_join_decision

,dataset,experimental_rows,participants,raw_log_files,P_in_workbook,sessionstart_in_workbook,P_in_raw_logs,join_status,supervised_use,dataset_role,reason
0,Dashboards,1008,63,50,0,0,0,NO_REPRODUCIBLE_JOIN,False,X_ONLY_AUXILIARY,No existe una clave observable que permita asociar cada log con P+DB+Q sin inferir correspondencias.


In [49]:
dashboards_behavior_policy = pd.DataFrame([
    {
        "canonical_variable": "click_count",
        "source_signal": "mouseup with left button",
        "derivable": True,
        "supervised_training": False,
        "note":
            "mouseup es candidato a clic; no sumar mousedown + mouseup."
    },
    {
        "canonical_variable": "keypress_count",
        "source_signal": "keypress",
        "derivable": True,
        "supervised_training": False,
        "note":
            "Número de eventos, no contenido de las teclas."
    },
    {
        "canonical_variable": "scroll_count",
        "source_signal": "scroll",
        "derivable": "REQUIRES_EPISODE_RULE",
        "supervised_training": False,
        "note":
            "Los eventos scroll son continuos; no contar cada evento como una acción."
    },
    {
        "canonical_variable": "mean_pre_action_pause_s",
        "source_signal": "timestampms before actions",
        "derivable": True,
        "supervised_training": False,
        "note":
            "Solo interpretable como intervalo temporal, no como confusión."
    },
    {
        "canonical_variable": "navigation_steps",
        "source_signal": "url/urlFull transitions",
        "derivable": True,
        "supervised_training": False,
        "note":
            "Requiere una definición explícita de cambio de estado."
    },
    {
        "canonical_variable": "backtracking_count",
        "source_signal": "revisited url/urlFull states",
        "derivable": True,
        "supervised_training": False,
        "note":
            "Revisita a un estado URL previo."
    },
    {
        "canonical_variable": "task_time_s",
        "source_signal": None,
        "derivable": False,
        "supervised_training": False,
        "note":
            "Sin join/tarea no se conocen límites de P+DB+Q."
    },
    {
        "canonical_variable": "session_duration_s",
        "source_signal": "max(timestampms)-min(timestampms)",
        "derivable": True,
        "supervised_training": False,
        "note":
            "Variable auxiliar; no equivale a task_time_s."
    }
])

dashboards_behavior_policy

,canonical_variable,source_signal,derivable,supervised_training,note
0,click_count,mouseup with left button,True,False,mouseup es candidato a clic; no sumar mousedown + mouseup.
1,keypress_count,keypress,True,False,"Número de eventos, no contenido de las teclas."
2,scroll_count,scroll,REQUIRES_EPISODE_RULE,False,Los eventos scroll son continuos; no contar cada evento como una acción.
3,mean_pre_action_pause_s,timestampms before actions,True,False,"Solo interpretable como intervalo temporal, no como confusión."
4,navigation_steps,url/urlFull transitions,True,False,Requiere una definición explícita de cambio de estado.
5,backtracking_count,revisited url/urlFull states,True,False,Revisita a un estado URL previo.
6,task_time_s,NaN,False,False,Sin join/tarea no se conocen límites de P+DB+Q.
7,session_duration_s,max(timestampms)-min(timestampms),True,False,Variable auxiliar; no equivale a task_time_s.


In [50]:
dashboards_join_decision.to_csv(
    REPORTS_DIR
    / "dashboards_join_decision.csv",
    index=False
)

dashboards_behavior_policy.to_csv(
    REPORTS_DIR
    / "dashboards_behavior_feature_policy_final.csv",
    index=False
)

In [51]:
for filename in [
    "dashboards_join_decision.csv",
    "dashboards_behavior_feature_policy_final.csv"
]:
    path = REPORTS_DIR / filename

    print(
        filename,
        "✅" if path.exists() else "❌"
    )

dashboards_join_decision.csv ✅
dashboards_behavior_feature_policy_final.csv ✅
